Eu tenho 100k newick na pasta mutation_dataset_100k_phylCNN

In [30]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from ete3 import Tree
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from sklearn.model_selection import train_test_split

In [31]:
TREE_DIR = r"mutation_dataset_100k_phyloCNN\\split_trees_corrected"
PARAM_FILE = r"mutation_dataset_100k_phyloCNN\\tabela_100k_phyloCNN.txt"

# BAD_INDICES = {
#    31035, 36174, 49680, 52494,
#    53227, 73155, 81488, 97073
#}

# no .nwk, a árvore dos ids de erro ta uma linha vazia, isso da problema
# acho melhor logo transformar o arquivo removendo as linhas vazias, e removendo os ids da tabela de parametros
# depois so precisa cuidar do v9, q vai 9992 arvores, mas qualquer coisa nbem treina dai

TARGET_COLUMNS = ["R_nought_1", "R_nought_2", "infectious_time"]

In [32]:
MUTANTS_FILE = r"mutation_dataset_100k_phyloCNN\\number_of_mutantes.npy"

#BAD_INDICES = [31035, 36174, 49680, 52494, 53227, 73155, 81488, 97073]
# Load parameter table
params = pd.read_csv(PARAM_FILE, sep="\t")

# Load number of mutants for each original tree
number_of_mutantes = np.load(MUTANTS_FILE)

# Check alignment before removing bad trees
if len(params) != len(number_of_mutantes):
    raise ValueError(
        f"Parameter table has {len(params)} rows, "
        f"but number_of_mutantes has {len(number_of_mutantes)} entries."
    )

# Add number of mutants
params["number_of_mutantes"] = number_of_mutantes


# Reset dataframe index
params.reset_index(drop=True, inplace=True)

# Total number of nodes in a full binary tree:
# total_nodes = 2 * internal_nodes + 1
params["total_nodes"] = 2 * params["tree_size"] + 1

# Fraction of mutated nodes
params["mutation_fraction"] = (
    params["number_of_mutantes"] / params["total_nodes"]
)


# <= 5 mutated nodes -> R02 is not reliably estimable
few_mutations = params["number_of_mutantes"] <= 5

params.loc[few_mutations, "R_nought_2"] = (
    params.loc[few_mutations, "R_nought_1"]
)

# Case 2:
# >95% of nodes mutated -> R01 is not reliably estimable
mostly_mutated = params["mutation_fraction"] > 0.95

params.loc[mostly_mutated, "R_nought_1"] = (
    params.loc[mostly_mutated, "R_nought_2"]
)

print("Original number of parameter rows:", len(number_of_mutantes))
#print("Removed bad trees:", len(BAD_INDICES))
print("Final parameter rows:", len(params))

print("Trees with <= 5 mutations:", few_mutations.sum())
print("Trees with >95% mutated nodes:", mostly_mutated.sum())

#print("\nBad indices still present:",
 #     sorted(set(params["index"]) & BAD_INDICES))


Original number of parameter rows: 99992
Final parameter rows: 99992
Trees with <= 5 mutations: 25985
Trees with >95% mutated nodes: 4962


In [33]:
params_by_id = params.set_index("index")

In [34]:
def ete_to_pyg(tree, target):
    nodes = list(tree.traverse())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    node_features = []
    edges = []

    for node in nodes:
        node_features.append([
            float(node.DIST_TO_START),
            float(node.is_leaf())
        ])

        if node.up is not None:
            parent = node_to_idx[node.up]
            child = node_to_idx[node]

            edges.append([parent, child])
            edges.append([child, parent])

    x = torch.tensor(node_features, dtype=torch.float32)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()



    #make target [1, 3], not [3]
    y = torch.tensor(target, dtype=torch.float32).unsqueeze(0)

    return Data(x=x, edge_index=edge_index, y=y)

In [35]:
def process_nwk(part_number, params):
    filename = os.path.join(TREE_DIR, f"trees_part_{part_number}.nwk")

    start_position = part_number * 10000
    end_position = start_position + 10000

    # Select the parameters corresponding to the trees in this file
    part_params = params.iloc[start_position:end_position].reset_index(drop=True)

    dataset = []

    with open(filename, "r") as f:
        tree_lines = [line.strip() for line in f if line.strip()]

    if len(tree_lines) != len(part_params):
        raise ValueError(
            f"Part {part_number}: "
            f"{len(tree_lines)} trees but "
            f"{len(part_params)} parameter rows."
        )

    for local_index, line in enumerate(tree_lines):

        # Original tree ID
        global_index = int(part_params.loc[local_index, "index"])

        #print(f"Processing tree {global_index} from part {part_number}")

        tree = Tree(line, format=1)

        target = part_params.loc[
            local_index, TARGET_COLUMNS
        ].to_numpy(dtype=np.float32)

        graph = ete_to_pyg(tree, target)
        graph.tree_id = global_index

        dataset.append(graph)

    print(f"Part {part_number}: {len(dataset)} valid trees")

    return dataset

In [36]:
dataset = process_nwk(7, params)

Part 7: 10000 valid trees


In [37]:
print(dataset[0])
print("Node features:", dataset[0].x.shape)
print(dataset[0].x[:5])

Data(x=[977, 2], edge_index=[2, 1952], y=[1, 3], tree_id=70005)
Node features: torch.Size([977, 2])
tensor([[ 5.8333,  0.0000],
        [ 8.9088,  0.0000],
        [11.0820,  1.0000],
        [11.6359,  0.0000],
        [14.9882,  0.0000]])


In [38]:
def split_dataset(dataset):
    indices = np.arange(len(dataset))

    train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    train_dataset = [dataset[i] for i in train_idx]
    val_dataset = [dataset[i] for i in val_idx]
    test_dataset = [dataset[i] for i in test_idx]

    return train_dataset, val_dataset, test_dataset

In [39]:
train_dataset, val_dataset, test_dataset = split_dataset(dataset)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

8000
1000
1000


In [40]:
def normalize_targets(dataset, target_mean, target_std):

    for graph in dataset:
        graph.y = (graph.y - target_mean)/target_std

    return dataset

In [41]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Descomentar no treino seguitne
checkpoint = torch.load("modelos\\GNN1\\v6.pt", map_location=device, weights_only=False)

target_mean = torch.tensor(checkpoint["target_mean"], dtype=torch.float32)

target_std = torch.tensor(checkpoint["target_std"], dtype=torch.float32)



In [42]:
train_y = torch.cat([g.y for g in train_dataset], dim=0)

target_mean = train_y.mean(dim=0)
target_std = train_y.std(dim=0).clamp_min(1e-8)

print("Target mean:", target_mean)
print("Target std:", target_std)

Target mean: tensor([3.1587, 5.8616, 5.4670])
Target std: tensor([1.4093, 3.1579, 2.6036])


In [43]:
train_dataset = normalize_targets(train_dataset, target_mean, target_std)
val_dataset = normalize_targets(val_dataset, target_mean, target_std)
test_dataset = normalize_targets(test_dataset, target_mean, target_std)

In [44]:
class PhyloGNN(nn.Module):
    def __init__(self, num_features=2):
        super().__init__()

        self.gnn1 = GATConv(num_features, 64, heads=4, concat=False)
        self.gnn2 = GATConv(64, 64, heads=4, concat=False)
        self.gnn3 = GATConv(64, 64, heads=4, concat=False)

        self.regressor = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

    def forward(self, x, edge_index, batch):
        x = torch.relu(self.gnn1(x, edge_index))
        x = torch.relu(self.gnn2(x, edge_index))
        x = torch.relu(self.gnn3(x, edge_index))

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)

        x = torch.cat([x_mean, x_max], dim=1)

        return self.regressor(x)

In [45]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [46]:
model = PhyloGNN(num_features=2).to(device)

#descomentar no treino seguinte
model.load_state_dict(checkpoint["model_state_dict"])

model.train()

PhyloGNN(
  (gnn1): GATConv(2, 64, heads=4)
  (gnn2): GATConv(64, 64, heads=4)
  (gnn3): GATConv(64, 64, heads=4)
  (regressor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=3, bias=True)
  )
)

In [47]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        pred = model(batch.x, batch.edge_index, batch.batch)

        loss = criterion(pred, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

In [48]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)

        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(pred, batch.y)

        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(loader.dataset)

In [49]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

In [50]:
batch = next(iter(train_loader))

batch = batch.to(device)

pred = model(batch.x, batch.edge_index, batch.batch)

print("pred:", pred.shape)
print("target:", batch.y.shape)

pred: torch.Size([32, 3])
target: torch.Size([32, 3])


In [51]:
best_val_loss = float("inf")

for epoch in range(1, 25):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch:03d} | Train: {train_loss:.5f} | Val: {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch

        torch.save({
                "model_state_dict": model.state_dict(),
                # These are the V0 normalization statistics.
                "target_mean": target_mean.cpu().numpy(),
                "target_std": target_std.cpu().numpy(),

                "best_val_loss": best_val_loss,
                "epoch": best_epoch}, "modelos\\GNN1\\v7.pt")

Epoch 001 | Train: 0.20928 | Val: 0.21227
Epoch 002 | Train: 0.20638 | Val: 0.22271
Epoch 003 | Train: 0.21503 | Val: 0.20201
Epoch 004 | Train: 0.19897 | Val: 0.18917
Epoch 005 | Train: 0.19367 | Val: 0.21201
Epoch 006 | Train: 0.19930 | Val: 0.19410
Epoch 007 | Train: 0.19738 | Val: 0.20829
Epoch 008 | Train: 0.19897 | Val: 0.23159
Epoch 009 | Train: 0.20037 | Val: 0.25383
Epoch 010 | Train: 0.20050 | Val: 0.20315
Epoch 011 | Train: 0.19430 | Val: 0.18808
Epoch 012 | Train: 0.19034 | Val: 0.21553
Epoch 013 | Train: 0.18864 | Val: 0.20249
Epoch 014 | Train: 0.19154 | Val: 0.18626
Epoch 015 | Train: 0.18624 | Val: 0.20090
Epoch 016 | Train: 0.18437 | Val: 0.20516
Epoch 017 | Train: 0.19435 | Val: 0.18964
Epoch 018 | Train: 0.19193 | Val: 0.18498
Epoch 019 | Train: 0.18336 | Val: 0.20441
Epoch 020 | Train: 0.19292 | Val: 0.19492
Epoch 021 | Train: 0.18264 | Val: 0.21798
Epoch 022 | Train: 0.18197 | Val: 0.18162
Epoch 023 | Train: 0.18509 | Val: 0.20050
Epoch 024 | Train: 0.18186 | Val: 

In [52]:
best_checkpoint = torch.load("modelos\\GNN1\\v7.pt", map_location=device, weights_only=False)

model.load_state_dict(best_checkpoint["model_state_dict"])

model.eval()

print("Best epoch:",best_checkpoint["epoch"])

print("Best validation loss:", best_checkpoint["best_val_loss"])

Best epoch: 22
Best validation loss: 0.18161853790283203


In [53]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()

    predictions = []
    true_values = []

    for batch in loader:
        batch = batch.to(device)

        pred = model(batch.x, batch.edge_index, batch.batch)

        predictions.append(pred.cpu().numpy())
        true_values.append(batch.y.cpu().numpy())

    return np.concatenate(predictions), np.concatenate(true_values)

In [54]:
pred_scaled, y_scaled = predict(model, test_loader, device)

In [55]:
mean_np = target_mean.cpu().numpy()
std_np = target_std.cpu().numpy()

pred = pred_scaled * std_np + mean_np
y_true = y_scaled * std_np + mean_np

In [56]:
from sklearn.metrics import mean_absolute_error

for i, name in enumerate(["R01", "R02", "Infectious time"]):
    mae = mean_absolute_error(y_true[:, i], pred[:, i])
    print(f"{name}: MAE = {mae:.4f}")

R01: MAE = 0.4009
R02: MAE = 1.2418
Infectious time: MAE = 0.4290


In [57]:
from sklearn.metrics import mean_squared_error

for i, name in enumerate(["R01", "R02", "Infectious time"]):
    rmse = mean_squared_error(y_true[:, i], pred[:, i]) ** 0.5
    print(f"{name}: RMSE = {rmse:.4f}")

R01: RMSE = 0.6372
R02: RMSE = 1.5968
Infectious time: RMSE = 0.5741


In [58]:
import numpy as np

mape = np.mean(
    np.abs((y_true - pred) / y_true),
    axis=0
) * 100

for i, name in enumerate(["R01", "R02", "Infectious time"]):
    print(f"{name}: MAPE = {mape[i]:.2f}%")

R01: MAPE = 13.89%
R02: MAPE = 26.08%
Infectious time: MAPE = 8.46%


In [59]:
n_positions = np.load(r"C:\Users\JPC\Documents\MESTRADO\Projeto\gillespie-2\mutation_dataset_100k_phyloCNN\mutation_positions.npy")
print(len(n_positions))

99992
